<a href="https://colab.research.google.com/github/crezny/DATASCI266_final_project/blob/main/DPO V3_7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
%pip install -q trl accelerate peft bitsandbytes datasets transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 465.5/465.5 kB 29.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.4/59.4 MB 45.4 MB/s eta 0:00:00


In [3]:
import pandas as pd
from datetime import datetime, time
from typing import List, Dict, Any
import pandas as pd
from typing import Optional, Iterable, Dict, Any, List
from datetime import datetime, time
from typing import List, Dict, Any
from dotenv import load_dotenv
from sqlalchemy import create_engine, text
from sqlalchemy.exc import SQLAlchemyError
from typing import Optional, Iterable, Dict, Any, List
import os
import numpy as np
import torch
from transformers import AutoModelForCausalLM
from peft import PeftModel

try:
    from tqdm.notebook import tqdm
except Exception:
    from tqdm import tqdm

try:
    from google.colab import userdata
    get_secret = userdata.get
except ImportError:
    # Fallback: define a dummy get_secret for local use
    def get_secret(key):
        return None

from dotenv import load_dotenv
load_dotenv()


from sqlalchemy import create_engine, text
from sqlalchemy.exc import SQLAlchemyError

def get_env(key: str, default: str = None):
    """Tries os.getenv first, then Colab userdata if available."""
    return os.getenv(key) or get_secret(key) or default

DB_HOST = get_env("POSTGRES_DB_HOST")
DB_PORT = int(get_env("POSTGRES_DB_PORT"))
DB_NAME = get_env("POSTGRES_DB_NAME")
DB_USER = get_env("POSTGRES_DB_USER")
DB_PASS = get_env("POSTGRES_DB_PASS")

connection_url = f"postgresql+psycopg2://{DB_USER}:{DB_PASS}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine = create_engine(connection_url, echo=False)



In [15]:
def get_dpo_df(split:str = "train") -> pd.DataFrame:
  v2_5_sql = f"""
    SELECT
      PROMPTS.*,
      SCORES."S_sem_faith",
      SCORES."S_surface",
      SCORES."S_ragas"
    FROM
      (
        SELECT
          PROMPT_RESPONSES.*,
          POLICY_SPLITS.SPLIT_LABEL,
          POLICY.ORG_TEXT
        FROM
          PROMPT_RESPONSES
          JOIN POLICY_SPLITS ON POLICY_SPLITS.POLICY_ID = PROMPT_RESPONSES.POLICY_ID
          JOIN (
            SELECT DISTINCT
              POLICY_ID,
              SECTION_ID,
              SECTION_TITLE,
              SOURCE_FRAMEWORK,
              STRING_AGG(
                CLAUSE_TEXT,
                '\n'
                ORDER BY
                  ROW_NUMBER
              ) AS ORG_TEXT
            FROM
              POLICY_LINES
            GROUP BY
              POLICY_ID,
              SECTION_ID,
              SECTION_TITLE,
              SOURCE_FRAMEWORK
          ) AS POLICY ON PROMPT_RESPONSES.SECTION_ID = POLICY.SECTION_ID
        WHERE
          VERSION = 'v2.5'
          AND SPLIT_LABEL = '{split}'
      ) AS PROMPTS
      JOIN (
        SELECT
          POLICY_ID,
          SECTION_ID,
          MODEL_NAME AS VERSION,
          (0.4 * "S_sem") + (0.4 * "S_ragas") + (0.2 * "S_ragas_relevancy") AS "S_sem_faith",
          (
            0.40 * "S_lex" + 0.30 * "S_len" + 0.30 * "S_spacy"
          ) AS "S_surface",
          "S_ragas"
        FROM
          SCORES6
      ) AS SCORES ON SCORES.SECTION_ID = PROMPTS.SECTION_ID
    WHERE
      SCORES.VERSION = 'v2.5'
    """
  v3_7_sql = f"""
    SELECT
      PROMPTS.*,
      SCORES."S_sem_faith",
      SCORES."S_surface",
      SCORES."S_ragas"
    FROM
      (
        SELECT
          PROMPT_RESPONSES.*,
          POLICY_SPLITS.SPLIT_LABEL,
          POLICY.ORG_TEXT
        FROM
          prompt_responses2 PROMPT_RESPONSES
          JOIN POLICY_SPLITS ON POLICY_SPLITS.POLICY_ID = PROMPT_RESPONSES.POLICY_ID
          JOIN (
            SELECT DISTINCT
              POLICY_ID,
              SECTION_ID,
              SECTION_TITLE,
              SOURCE_FRAMEWORK,
              STRING_AGG(
                CLAUSE_TEXT,
                '\n'
                ORDER BY
                  ROW_NUMBER
              ) AS ORG_TEXT
            FROM
              POLICY_LINES
            GROUP BY
              POLICY_ID,
              SECTION_ID,
              SECTION_TITLE,
              SOURCE_FRAMEWORK
          ) AS POLICY ON PROMPT_RESPONSES.SECTION_ID = POLICY.SECTION_ID
        WHERE
          VERSION = 'v3.7'
          AND PROMPT_RESPONSES.SPLIT_LABEL = '{split}'
      ) AS PROMPTS
      JOIN (
        SELECT
          POLICY_ID,
          SECTION_ID,
          MODEL_NAME AS VERSION,
          (0.4 * "S_sem") + (0.4 * "S_ragas") + (0.2 * "S_ragas_relevancy") AS "S_sem_faith",
          (
            0.40 * "S_lex" + 0.30 * "S_len" + 0.30 * "S_spacy"
          ) AS "S_surface",
          "S_ragas"
        FROM
          SCORES6
      ) AS SCORES ON SCORES.SECTION_ID = PROMPTS.SECTION_ID
    WHERE
      SCORES.VERSION = 'v3.7'
    """

  with engine.connect() as conn:
      df_25 = pd.read_sql(text(v2_5_sql), conn)
      df_37 = pd.read_sql(text(v3_7_sql), conn)
  USE_RAGAS = True           # Toggle RAGAS on/off in the preference score
  ALPHA_SEM = 0.7            # Weight on semantic faithfulness vs surface when RAGAS is off
  MARGIN = 0.08              # Min delta in score to keep a pair (absolute difference)
  MIN_SEM_FLOOR   = 0.35     # at least one answer must be this good on S_sem_faith
  MIN_SCORE_FLOOR = -0.5     # drop pairs where both scores are very low (even if different)
  RAGAS_COLS = ["S_ragas"]

  # Weights when RAGAS is ON
  W_SEM = 0.8
  W_SURF = 0.2

  key_cols = ["policy_id", "section_id"]

  base_cols_needed = key_cols + [
      "system_prompt",
      "user_prompt",
      "output",
      "S_sem_faith",
      "S_surface",
  ]

  ragas_cols_present = [c for c in RAGAS_COLS if c in df_25.columns and c in df_37.columns]
  cols_needed = base_cols_needed + ragas_cols_present

  df25 = df_25[cols_needed].rename(columns={
      "system_prompt": "system_prompt_v25",
      "user_prompt": "user_prompt_v25",
      "output": "resp_v25",
      "S_sem_faith": "S_sem_faith_v25",
      "S_surface": "S_surface_v25",
      **{c: f"{c}_v25" for c in ragas_cols_present},
  }).copy()

  df37 = df_37[cols_needed].rename(columns={
      "system_prompt": "system_prompt_v37",
      "user_prompt": "user_prompt_v37",
      "output": "resp_v37",
      "S_sem_faith": "S_sem_faith_v37",
      "S_surface": "S_surface_v37",
      **{c: f"{c}_v37" for c in ragas_cols_present},
  }).copy()

  pairs = (
      df25
      .merge(df37, on=key_cols, how="inner")
      .dropna(subset=["resp_v25", "resp_v37"])
  )
  mismatch_mask = (
      (pairs["system_prompt_v25"] != pairs["system_prompt_v37"]) |
      (pairs["user_prompt_v25"]   != pairs["user_prompt_v37"])
  )
  if mismatch_mask.any():
      print("Warning: some system/user prompts differ between v2.5 and v3.7")


  def format_prompt(sys, usr):
      sys = sys or ""
      usr = usr or ""

      return f"<|system|>\n{sys}\n\n<|user|>\n{usr}"

  pairs["prompt"] = pairs.apply(
      lambda r: format_prompt(r["system_prompt_v25"], r["user_prompt_v25"]),
      axis=1
  )




  metrics_to_norm = [
      "S_sem_faith_v25", "S_sem_faith_v37",
      "S_surface_v25",   "S_surface_v37",
  ]

  for col in metrics_to_norm:
      mu = pairs[col].mean()
      sigma = pairs[col].std(ddof=1)
      if sigma == 0 or np.isnan(sigma):
          sigma = 1.0
      pairs[col + "_z"] = (pairs[col] - mu) / sigma


  def score_row(row, w_sem=0.8, w_surf=0.2):
      sem_v25  = row["S_sem_faith_v25_z"]
      sem_v37  = row["S_sem_faith_v37_z"]
      surf_v25 = row["S_surface_v25_z"]
      surf_v37 = row["S_surface_v37_z"]

      score_v25 = w_sem * sem_v25 + w_surf * surf_v25
      score_v37 = w_sem * sem_v37 + w_surf * surf_v37

      return score_v25, score_v37


  pairs[["score_v25", "score_v37"]] = pairs.apply(
      lambda r: pd.Series(score_row(r)),
      axis=1
  )



  quality_cols = [
    "S_sem_faith_v25", "S_sem_faith_v37",
    "S_surface_v25",   "S_surface_v37",
  ]


  pairs = pairs.dropna(subset=quality_cols)

  # --- 2) Require at least ONE reasonably good answer per pair ---
  # best semantic faithfulness across the two versions
  best_sem = pairs[["S_sem_faith_v25", "S_sem_faith_v37"]].max(axis=1)
  good_sem_mask = best_sem >= MIN_SEM_FLOOR

  good_ragas_mask = True

  pairs = pairs[good_sem_mask & good_ragas_mask]

  print("After semantic/RAGAS quality filter:", len(pairs))

  # --- 3) Confidence filter by margin on the COMBINED score ---
  pairs["delta"] = pairs["score_v37"] - pairs["score_v25"]  # >0 => v3.7 scored better

  # drop pairs where both versions are just bad even if different
  best_score = pairs[["score_v25", "score_v37"]].max(axis=1)
  non_garbage_mask = best_score >= MIN_SCORE_FLOOR

  # require a meaningful gap in score
  margin_mask = np.abs(pairs["delta"]) >= MARGIN

  confident = pairs[non_garbage_mask & margin_mask].copy()

  print("Total pairs before filters:", len(pairs))
  print("Total pairs after  filters:", len(confident))

  # --- 4) (Optional) ensure the preference isn’t fighting semantic faithfulness ---
  # If combined score says A>B but semantic_z says the opposite with a big margin,
  # treat the pair as ambiguous and drop it.
  sem_delta = confident["S_sem_faith_v37_z"] - confident["S_sem_faith_v25_z"]
  score_delta = confident["score_v37"] - confident["score_v25"]
  agree_mask = np.sign(sem_delta) == np.sign(score_delta)

  # keep only pairs where semantic direction agrees with overall preference
  confident = confident[agree_mask].copy()
  print("After semantic-agreement filter:", len(confident))


  # --- 5) Build chosen/rejected using shared prompt ---
  def choose_pair(row):
      if row["score_v25"] > row["score_v37"]:
          chosen   = row["resp_v25"]
          rejected = row["resp_v37"]
      else:
          chosen   = row["resp_v37"]
          rejected = row["resp_v25"]

      return pd.Series({
          "prompt":   row["prompt"],
          "chosen":   chosen,
          "rejected": rejected,
      })

  dpo_df = confident.apply(choose_pair, axis=1).reset_index(drop=True)
  return dpo_df




In [11]:
val_dpo_df = get_dpo_df(split="val")

After semantic/RAGAS quality filter: 113
Total pairs before filters: 113
Total pairs after  filters: 99
After semantic-agreement filter: 97


In [16]:
import os
train_dpo_df = get_dpo_df(split="train")
val_dpo_df = get_dpo_df(split="val")


from datasets import Dataset, DatasetDict

dataset = DatasetDict({
    "train": Dataset.from_pandas(train_dpo_df),
    "val": Dataset.from_pandas(val_dpo_df)
})




After semantic/RAGAS quality filter: 1458
Total pairs before filters: 1458
Total pairs after  filters: 1255
After semantic-agreement filter: 1180
After semantic/RAGAS quality filter: 117
Total pairs before filters: 117
Total pairs after  filters: 98
After semantic-agreement filter: 86


In [7]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [17]:
import torch
from transformers import AutoModelForCausalLM
from peft import PeftModel
from transformers import AutoTokenizer

base_model_name = "google/gemma-2-9b-it"
lora_dir = "/content/drive/MyDrive/266/gemma2_9b_it_v2_5_lora"

tokenizer = AutoTokenizer.from_pretrained(base_model_name)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"
tokenizer.truncation_side = "right"


policy_base = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    dtype=torch.bfloat16,
    device_map="auto",
)
policy_model = PeftModel.from_pretrained(
    policy_base,
    lora_dir,
)

policy_model.print_trainable_parameters()
ref_model = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

trainable params: 0 || all params: 9,295,724,032 || trainable%: 0.0000


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

In [18]:
from trl import DPOTrainer, DPOConfig
MAX_LENGTH = 2096

output_dir = "/content/drive/MyDrive/266/gemma2_9b_it_v2_5_lora_dpo_2"

dpo_args = DPOConfig(
    output_dir=output_dir,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=5e-6,
    num_train_epochs=1.0,
    bf16=True,
    logging_steps=10,

    save_strategy="steps",
    save_steps=200,

    beta=0.1,
    max_length=MAX_LENGTH,
    max_prompt_length=MAX_LENGTH // 2,

    report_to="none",
)

trainer = DPOTrainer(
    model=policy_model,
    ref_model=ref_model,
    args=dpo_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["val"],
    processing_class=tokenizer,
)

trainer.train()



Extracting prompt in train dataset:   0%|          | 0/1180 [00:00<?, ? examples/s]

Applying chat template to train dataset:   0%|          | 0/1180 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/1180 [00:00<?, ? examples/s]

Extracting prompt in eval dataset:   0%|          | 0/86 [00:00<?, ? examples/s]

Applying chat template to eval dataset:   0%|          | 0/86 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/86 [00:00<?, ? examples/s]

The model is already on multiple devices. Skipping the move to device specified in `args`.
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 1}.


Step,Training Loss
10,5.994200
20,4.944300
30,5.973300
40,5.508700
50,3.868800
60,4.370700
70,5.524000
80,5.025300
90,5.360500
100,5.241100


TrainOutput(global_step=148, training_loss=5.510654578337798, metrics={'train_runtime': 1788.621, 'train_samples_per_second': 0.66, 'train_steps_per_second': 0.083, 'total_flos': 0.0, 'train_loss': 5.510654578337798, 'epoch': 1.0})

In [19]:
merged = policy_model.merge_and_unload()
merged_dir = os.path.join(output_dir, "gemma_merged_full_model_2")
merged.save_pretrained(merged_dir)
tokenizer.save_pretrained(merged_dir)


('/content/drive/MyDrive/266/gemma2_9b_it_v2_5_lora_dpo_2/gemma_merged_full_model_2/tokenizer_config.json',
 '/content/drive/MyDrive/266/gemma2_9b_it_v2_5_lora_dpo_2/gemma_merged_full_model_2/special_tokens_map.json',
 '/content/drive/MyDrive/266/gemma2_9b_it_v2_5_lora_dpo_2/gemma_merged_full_model_2/chat_template.jinja',
 '/content/drive/MyDrive/266/gemma2_9b_it_v2_5_lora_dpo_2/gemma_merged_full_model_2/tokenizer.model',
 '/content/drive/MyDrive/266/gemma2_9b_it_v2_5_lora_dpo_2/gemma_merged_full_model_2/added_tokens.json',
 '/content/drive/MyDrive/266/gemma2_9b_it_v2_5_lora_dpo_2/gemma_merged_full_model_2/tokenizer.json')

In [ ]:
# import shutil
# shutil.make_archive(merged_dir, 'zip', merged_dir)

In [ ]:
for rec in trainer.state.log_history:
    if 'loss' in rec:
        print(rec['step'], rec['loss'])

10 5.0401
20 5.1419
30 6.3132
40 5.396
50 7.3769
60 4.4345
70 4.4874
80 4.9298
90 5.7234
100 5.5358
110 5.1248
120 5.6926
130 5.4702
140 5.7581
150 5.3391
160 6.9661
170 4.761
